# cAPTure: graph logit tail and threshold-scope audit

This audit investigates why ST-GNN has strong within-scenario ranking but very low recall at the frozen global false-alert budget. It uses the existing development OOF logits and prepared packet metadata. Fold-local and scenario-local thresholds are in-sample diagnostics only; the frozen global threshold remains the operational result. No GPU, checkpoint, or training is required.


## 1. Prepare Colab and mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import subprocess
import sys

REPOSITORY_URL = "https://github.com/tatipar/temporalgnn-nids.git"
REPOSITORY_BRANCH = "feat/capture-feasibility"
PROJECT_ROOT = Path("/content/temporalgnn-nids")
DRIVE_ROOT = Path("/content/drive/MyDrive/capture_gate0")

if not PROJECT_ROOT.exists():
    subprocess.run(["git", "clone", "--branch", REPOSITORY_BRANCH,
                    "--single-branch", REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
else:
    subprocess.run(["git", "pull", "--ff-only", "origin",
                    REPOSITORY_BRANCH], cwd=PROJECT_ROOT, check=True)
branch = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=PROJECT_ROOT, text=True
).strip()
if branch != REPOSITORY_BRANCH:
    raise RuntimeError(f"Expected branch {REPOSITORY_BRANCH}, found {branch}.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(PROJECT_ROOT / "code/python/requirements-capture-mlp.txt")],
               check=True)
sys.path.insert(0, str(PROJECT_ROOT / "code/python"))

import pandas as pd
from IPython.display import Markdown, display
from utils.capture_graph_logit_tail_audit import (
    run_logit_tail_audit, validate_logit_tail_audit,
)
print("Repository commit:", subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip())
print("CPU runtime is sufficient.")

## 2. Bind immutable development inputs

In [ ]:
RESCORING_RUN_ID = "20261001_graph_oof_logit_rescoring_v1"
COMPARISON_RUN_ID = "20261001_graph_logit_comparison_v1"
PREPARED_RUN_ID = "20260917T235058_827743Z_prepare_full_dev"
AUDIT_RUN_ID = "20261002_graph_logit_tail_audit_v1"

AUDIT_ARGUMENTS = {
    "config_path": PROJECT_ROOT / "configs/capture_graph_logit_tail_audit_v1.yaml",
    "comparison_dir": DRIVE_ROOT / "graph_logit_comparison_runs" / COMPARISON_RUN_ID,
    "rescoring_dir": DRIVE_ROOT / "graph_oof_rescoring_runs" / RESCORING_RUN_ID,
    "prepared_dir": DRIVE_ROOT / "prepared_runs" / PREPARED_RUN_ID,
    "output_dir": DRIVE_ROOT / "graph_logit_tail_audit_runs" / AUDIT_RUN_ID,
}
for name, path in AUDIT_ARGUMENTS.items():
    if name != "output_dir" and not path.exists():
        raise FileNotFoundError(f"Missing {name}: {path}")
print("Comparison input:", AUDIT_ARGUMENTS["comparison_dir"])
print("Rescoring input:", AUDIT_ARGUMENTS["rescoring_dir"])
print("Audit output:", AUDIT_ARGUMENTS["output_dir"])

## 3. Run or validate the audit

Review the paths above and set `RUN_AUDIT=True`. The audit reads the existing OOF files and saves a new immutable report in Drive. Reopening a completed run validates its hashes.

In [ ]:
RUN_AUDIT = False

if AUDIT_ARGUMENTS["output_dir"].exists():
    audit_report = validate_logit_tail_audit(**AUDIT_ARGUMENTS)
elif RUN_AUDIT:
    audit_report = run_logit_tail_audit(**AUDIT_ARGUMENTS)
else:
    audit_report = None
    print("Audit disabled. Set RUN_AUDIT=True to create the report.")

## 4. Inspect score scales and false-alert tails

In [ ]:
if audit_report is None:
    print("Run section 3 before reviewing results.")
else:
    print("Audit status:", audit_report["status"])
    display(pd.DataFrame(audit_report["fold_scale_summary"]))
    display(pd.DataFrame(audit_report["packet_logit_tails"]))
    display(pd.DataFrame(audit_report["window_logit_tails"]))
    display(pd.DataFrame(audit_report["threshold_scope_metrics"]))
    display(pd.DataFrame(audit_report["st_gnn_top_window_metadata"]))
    display(Markdown((AUDIT_ARGUMENTS["output_dir"] / "review.md").read_text(encoding="utf-8")))

## 5. Provenance boundary

In [ ]:
if audit_report is not None:
    print("Frozen global threshold unchanged:", audit_report["global_threshold_unchanged"])
    print("Local thresholds are in-sample diagnostics:", audit_report["local_thresholds_are_in_sample_diagnostics"])
    print("Model training performed:", audit_report["model_training_performed"])
    print("Checkpoint loading performed:", audit_report["checkpoint_loading_performed"])
    print("Held-out scenarios accessed:", audit_report["held_out_scenarios_accessed"])
    print("Saved audit:", AUDIT_ARGUMENTS["output_dir"] / "audit_report.json")